# Research Notebook: Vicious Circle Analysis

## About
Notebook for clustering and correlation analysis on socio-economic vulnerability and disability dataset.

---

## Notebook Structure

### Setup (run first!)
| Cell | Description |
|------|-------------|
| Imports | Load libraries |
| Environment | Mount Drive (Colab) or set local path |
| Functions | Data loading & processing functions |
| Load Data | Load and merge all datasets |
| Rates & Indices | Calculate rates, build health indices by age group |
| Data Prep | Create working dataset |

### Cluster Analysis (K-Means)
Discover natural groupings using Elbow Method + K-Means.

### Correlation Visualizations
Explore relationships between social vulnerability and health outcomes.

---

## Key Variables
- `social_index` — composite vulnerability (higher = worse)
- `working_age_health_index` — disability burden ages 18-64 (higher = worse)
- `elderly_health_index` — long-term care burden ages 65+ (higher = worse)
- `child_health_index` — disabled children rate ages 0-17 (higher = worse)
- `total_population` — settlement size

In [60]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import plotly.express as px
import plotly.graph_objects as go

# For OLS trendlines in plotly
try:
    import statsmodels
except ImportError:
    print("Warning: statsmodels not installed. Trendlines may not work.")
    print("Install with: pip install statsmodels")

In [61]:
# === ENVIRONMENT SETUP ===
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_PATH = "/content/drive/MyDrive/datas_for_research_vicious_circle_project (1)"
else:
    # Local path - adjust if needed
    BASE_PATH = r"C:\Users\kiril\OneDrive\Desktop\My_projects\TovTechOrg\bituah_da\datas_for_research_vicious_circle_project"

os.chdir(BASE_PATH)
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\kiril\OneDrive\Desktop\My_projects\TovTechOrg\bituah_da\datas_for_research_vicious_circle_project


In [62]:
# === DATA LOADING FUNCTIONS ===

def load_data(paths):
    return {
        "benefits": pd.read_excel(paths["benefits"], header=None),
        "lamas": pd.read_excel(paths["lamas"], sheet_name="נתונים פיזיים ונתוני אוכלוסייה ", header=None),
        "socio_regional": pd.read_excel(paths["socio_regional"], header=None),
        "periph_regional": pd.read_excel(paths["periph_regional"], header=None),
        "coordinates": pd.read_csv(paths["coordinates"])
    }

def merge_lamas(df_benefits, df_lamas):
    before = len(df_benefits)
    df = df_benefits.merge(
        df_lamas[["settlement_symbol", "socio_economic_index_cluster",
                  "socio_economic_index_score", "peripherality_index_cluster",
                  "peripherality_index_score"]],
        on="settlement_symbol", how="left"
    )
    assert len(df) == before, f"Lost rows in LAMAS merge: {before} -> {len(df)}"
    return df

def merge_index_from_regional(df_main, df_regional, index_cols, key="settlement_symbol"):
    before = len(df_main)
    df = df_main.merge(df_regional[[key] + index_cols], on=key, how="left", suffixes=("", "_reg"))
    assert len(df) == before, f"Row count mismatch: {before} -> {len(df)}"
    for col in index_cols:
        df[col] = df[col].combine_first(df[f"{col}_reg"])
    df = df.drop(columns=[f"{col}_reg" for col in index_cols])
    return df

def merge_coordinates(df_main, df_coordinates):
    before = len(df_main)
    df = df_main.merge(
        df_coordinates[["settlement_code", "lat", "lon"]],
        left_on="settlement_symbol", right_on="settlement_code", how="left"
    )
    df = df.drop(columns=["settlement_code"])
    assert len(df) == before, f"Lost rows in COORDINATES merge: {before} -> {len(df)}"
    return df

def clean_values(df):
    df = df.replace("***", np.nan)
    last_4_cols = df.columns[-4:]
    df = df.dropna(subset=last_4_cols, how="all")

    cols_to_drop = ["settlement_type", "injury_allowance", "recipients_of_the_senior_citizen_pension_only",
                    "recipients_of_the_pension_with_income_supplementation",
                    "total_recipients_of_old_age_and/or_survivors'_benefits",
                    "num_families_receiving_child_benefit", "num_children_receiving_child_benefit",
                    "families_with_4+_children_receiving_child_benefit", "maternity_benefits", "alimony"]
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    categorial_cols = ["socio_economic_index_cluster", "peripherality_index_cluster"]
    numeric_cols = df.loc[:, "total_population":"unemployment_benefit"].columns
    float_cols = ["socio_economic_index_score", "peripherality_index_score", "lon", "lat"]

    for col in df.columns:
        if col in categorial_cols:
            df[col] = df[col].astype("category")
        elif col in numeric_cols:
            df[col] = df[col].astype(str).str.replace(",", "", regex=False)
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)
        elif col in float_cols:
            df[col] = df[col].astype(float)
        else:
            df[col] = df[col].astype("object")
    return df

print("Functions loaded!")

Functions loaded!


In [63]:
# === LOAD AND PROCESS DATA ===

paths = {
    "benefits": "benefits_2024_12.xlsx",
    "lamas": "p_libud_23.xlsx",
    "socio_regional": "24_24_230t3.xlsx",
    "periph_regional": "24_22_420t3.xlsx",
    "coordinates": "israel_settlements_all_with_coords.csv"
}

dfs = load_data(paths)

# Process benefits
df_benefits = dfs["benefits"].iloc[5:].copy().reset_index(drop=True)
df_benefits.columns = [
    "settlement_name", "settlement_symbol", "settlement_type",
    "total_population", "population_0_17", "population_18_64", "population_65_plus",
    "total_recipients_of_old_age_and/or_survivors'_benefits",
    "recipients_of_the_pension_with_income_supplementation",
    "recipients_of_the_senior_citizen_pension_only",
    "long_term_care_benefit", "general_disability_benefit",
    "special_services_for_persons_with_disabilities", "disabled_child_benefit",
    "mobility_benefit", "work_injury_victims_receiving_disability_and_dependents'_benefits",
    "injury_allowance", "num_families_receiving_child_benefit",
    "num_children_receiving_child_benefit", "families_with_4+_children_receiving_child_benefit",
    "maternity_benefits", "alimony", "income_support_benefit", "unemployment_benefit"
]

# Process LAMAS
df_lamas = dfs["lamas"].iloc[9:].copy().reset_index(drop=True)
df_lamas = df_lamas[df_lamas[3] != 'מועצה אזורית']
df_lamas.rename(columns={1: "settlement_symbol", 250: "socio_economic_index_cluster",
                         251: "socio_economic_index_score", 256: "peripherality_index_cluster",
                         257: "peripherality_index_score"}, inplace=True)

# Process regional data
df_socio = dfs["socio_regional"].iloc[10:].copy().reset_index(drop=True).iloc[:-8]
df_socio.rename(columns={5: "settlement_symbol", 12: "socio_economic_index_cluster",
                         10: "socio_economic_index_score"}, inplace=True)

df_periph = dfs["periph_regional"].iloc[9:].copy().reset_index(drop=True).iloc[:-4]
df_periph.rename(columns={4: "settlement_symbol", 12: "peripherality_index_cluster",
                          10: "peripherality_index_score"}, inplace=True)

# Merge all
data_master = merge_lamas(df_benefits, df_lamas)
data_master = merge_index_from_regional(data_master, df_socio,
    index_cols=["socio_economic_index_cluster", "socio_economic_index_score"])
data_master = merge_index_from_regional(data_master, df_periph,
    index_cols=["peripherality_index_cluster", "peripherality_index_score"])
data_master = merge_coordinates(data_master, dfs["coordinates"])
data_master = clean_values(data_master)

print(f"Loaded {len(data_master)} settlements")

Loaded 278 settlements


In [64]:
# === CALCULATE RATES AND INDICES (IMPROVED) ===

# 1. Calculate rates
data_master['general_disability_rate'] = (data_master['general_disability_benefit'] / data_master['population_18_64'] * 100).round(2)
data_master['special_services_disability_rate'] = (data_master['special_services_for_persons_with_disabilities'] / data_master['population_18_64'] * 100).round(2)
data_master['mobility_disability_rate'] = (data_master['mobility_benefit'] / data_master['population_18_64'] * 100).round(2)
data_master['long_term_care_rate'] = (data_master['long_term_care_benefit'] / data_master['population_65_plus'] * 100).round(2)
data_master['disabled_child_rate'] = (data_master['disabled_child_benefit'] / data_master['population_0_17'] * 100).round(2)

# 2. Social Index (clean - only exogenous factors)
social_cols = ['socio_economic_index_score', 'peripherality_index_score']
df_social = data_master[social_cols].copy()
df_social['socio_economic_index_score'] *= -1
df_social['peripherality_index_score'] *= -1
data_master['social_index'] = StandardScaler().fit_transform(df_social).mean(axis=1)

# 3. Health Indices (split by age group)
working_age_cols = ['general_disability_rate', 'special_services_disability_rate', 'mobility_disability_rate']
data_master['working_age_health_index'] = StandardScaler().fit_transform(data_master[working_age_cols]).mean(axis=1)

elderly_cols = ['long_term_care_rate']
data_master['elderly_health_index'] = StandardScaler().fit_transform(data_master[elderly_cols]).mean(axis=1)

child_cols = ['disabled_child_rate']
data_master['child_health_index'] = StandardScaler().fit_transform(data_master[child_cols]).mean(axis=1)

print("Indices calculated!")

Indices calculated!


## Data Preparation for Analysis

Create working dataset with key variables for clustering and correlation analysis.

In [65]:
df_work = data_master[[
    "settlement_name",
    "social_index",
    "working_age_health_index",
    "elderly_health_index",
    "child_health_index",
    "total_population"
]].copy()

df_work = df_work.dropna()
df_work["log_population"] = np.log1p(df_work["total_population"])
df_work["pct_elderly"] = data_master.loc[df_work.index, "population_65_plus"] / data_master.loc[df_work.index, "total_population"]
df_work["pct_children"] = data_master.loc[df_work.index, "population_0_17"] / data_master.loc[df_work.index, "total_population"]

print(f"Total settlements: {len(df_work)}")


Total settlements: 232


## Cluster Analysis (K-Means)

### The Elbow Method
To find the optimal number of clusters (K):
1. Run K-Means for K = 1, 2, 3, ... 10
2. Calculate **inertia** (within-cluster sum of squares)
3. Plot inertia vs K — look for the "elbow" where adding more clusters stops helping much

### Features for Clustering
- `social_index` — socio-economic vulnerability
- `working_age_health_index` — disability/health burden (ages 18-64)
- `child_health_index` — disabled children rate (ages 0-17)
- `pct_elderly` — share of population 65+
- `log_population` — settlement size

In [66]:
# === CLUSTERING: ELBOW METHOD + K-MEANS ===

# Features for clustering
cluster_features = [
    "social_index",
    "working_age_health_index",
    "child_health_index",
    "pct_elderly",
    "log_population"
]
X_cluster = df_work[cluster_features].dropna().to_numpy()

# Standardize
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# === ELBOW METHOD ===
k_range = range(1, 11)
inertias = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertias.append(kmeans.inertia_)

# Plot Elbow
fig_elbow = px.line(
    x=list(k_range),
    y=inertias,
    markers=True,
    labels={"x": "Number of Clusters (K)", "y": "Inertia (Within-cluster SS)"},
    title="Elbow Method: Finding Optimal Number of Clusters"
)

fig_elbow.update_traces(marker=dict(size=10), line=dict(width=3))
fig_elbow.update_layout(
    title_font_size=18,
    xaxis=dict(tickmode="linear", tick0=1, dtick=1),
    plot_bgcolor="white"
)
fig_elbow.update_xaxes(showgrid=True, gridcolor="lightgray")
fig_elbow.update_yaxes(showgrid=True, gridcolor="lightgray")

fig_elbow.show()

# === APPLY CLUSTERING WITH OPTIMAL K ===
OPTIMAL_K = 3  # Based on elbow - change if needed

kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df_work["cluster"] = kmeans_final.fit_predict(X_cluster_scaled)

print(f"\nApplied K-Means with K={OPTIMAL_K}")
print(f"Cluster distribution:\n{df_work['cluster'].value_counts().sort_index()}")

c:\Users\kiril\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\kiril\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\kiril\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\kiril\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than 

c:\Users\kiril\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.




Applied K-Means with K=3
Cluster distribution:
cluster
0     61
1    108
2     63
Name: count, dtype: int64


In [67]:
# === CLUSTER SUMMARY ===

cluster_summary = df_work.groupby("cluster").agg({
    "social_index": "mean",
    "working_age_health_index": "mean",
    "child_health_index": "mean",
    "elderly_health_index": "mean",
    "pct_elderly": "mean",
    "pct_children": "mean",
    "total_population": ["mean", "count"]
}).round(3)

print(f"=== Cluster Summary (K={OPTIMAL_K}) ===\n")
print(cluster_summary)

=== Cluster Summary (K=3) ===

        social_index working_age_health_index child_health_index  \
                mean                     mean               mean   
cluster                                                            
0             -0.717                   -0.869             -0.166   
1              0.618                    0.479             -0.344   
2             -0.306                    0.366              1.354   

        elderly_health_index pct_elderly pct_children total_population        
                        mean        mean         mean             mean count  
cluster                                                                       
0                     -0.915       0.119        0.324        15416.295    61  
1                      0.894       0.064        0.353        15355.250   108  
2                     -0.215       0.158        0.292        98823.937    63  


In [68]:
# === NAME CLUSTERS BASED ON THEIR PROFILE ===

cluster_means = df_work.groupby("cluster")[["social_index", "working_age_health_index", "child_health_index"]].mean()
print(cluster_means)

# === VISUALIZE CLUSTERS ===

fig_clusters = px.scatter(
    df_work,
    x="social_index",
    y="working_age_health_index",
    color=df_work["cluster"].astype(str),
    hover_name="settlement_name",
    size="total_population",
    size_max=40,
    hover_data=["child_health_index", "elderly_health_index", "pct_elderly", "pct_children"],
    color_discrete_sequence=px.colors.qualitative.Set1,
    labels={
        "social_index": "Social Vulnerability Index",
        "working_age_health_index": "Working-Age Disability Index (18-64)",
        "child_health_index": "Child Disability Index (0-17)",
        "elderly_health_index": "Elderly Care Index (65+)",
        "pct_elderly": "% Elderly (65+)",
        "pct_children": "% Children (0-17)",
        "color": "Cluster"
    },
    title=f"Settlement Clusters (K={OPTIMAL_K})"
)

fig_clusters.update_layout(
    plot_bgcolor="white",
    width=1100,
    height=800,
    annotations=[
        dict(
            text=(
                "<b>How to read:</b><br>"
                "Social Index: higher = more vulnerable<br>"
                "Working-Age Health: higher = more disability (18-64)<br>"
                "Child Health: higher = more child disability (future risk)"
            ),
            xref="paper", yref="paper",
            x=0.02, y=0.98,
            showarrow=False,
            align="left",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=12)
        )
    ]
)

fig_clusters.show()

         social_index  working_age_health_index  child_health_index
cluster                                                            
0           -0.717326                 -0.869156           -0.165852
1            0.618295                  0.479003           -0.343593
2           -0.305935                  0.366227            1.353764


## Research Findings: Vicious Circle Analysis

### Key Discovery: The Demographic Time Bomb

| Cluster | Description | Working-Age Health | Child Health | Trend |
|---------|-------------|-------------------|--------------|-------|
| 0 | Prosperous small towns | -0.869 (healthy) | -0.166 | Stable |
| 1 | Peripheral settlements | +0.479 (high disability) | -0.344 | ✅ Improving |
| 2 | Large cities | +0.366 (medium) | **+1.354** (very high) | ⚠️ CRITICAL |

### Key Correlations

- `social_index ↔ working_age_health: r=0.449` — Vicious circle confirmed
- `social_index ↔ child_health: r=-0.196` — Poor children appear healthier (underdiagnosis?)
- `pct_elderly ↔ child_health: r=0.474` — Aging towns have sicker children

### Vicious Circle by Cluster

| Cluster | Correlation | Interpretation |
|---------|-------------|----------------|
| 0 | r = -0.165 | No vicious circle (prosperous) |
| 1 | r = -0.013 | No correlation |
| 2 | r = +0.438 | **Vicious circle confirmed** |

### Insights

1. **Cluster 2 (Large cities)** — highest child disability rate despite medium social vulnerability. This is the "demographic time bomb" — these children will become working-age adults with disabilities.

2. **Cluster 1 (Peripheral)** — high current disability but LOW child disability suggests situation is IMPROVING.

3. **The vicious circle hypothesis only holds in large cities (Cluster 2)**, not in smaller settlements.

In [69]:
# === CORRELATION HEATMAP ===

corr_cols = ["social_index", "working_age_health_index", "child_health_index",
             "elderly_health_index", "pct_elderly", "pct_children"]

corr_matrix = df_work[corr_cols].corr().round(3)

fig_heatmap = px.imshow(
    corr_matrix,
    labels=dict(color="Correlation"),
    x=["Social", "Working-Age Health", "Child Health", "Elderly Health", "% Elderly", "% Children"],
    y=["Social", "Working-Age Health", "Child Health", "Elderly Health", "% Elderly", "% Children"],
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    text_auto=True
)

fig_heatmap.update_layout(
    title="Correlation Matrix: Social, Health & Demographics",
    width=800,
    height=700
)

fig_heatmap.show()

In [70]:
# --- 2.2 Main Relationship: Social vs Working-Age Health ---

fig_main = px.scatter(
    df_work,
    x="social_index",
    y="working_age_health_index",
    color=df_work["cluster"].astype(str),
    size="total_population",
    hover_name="settlement_name",
    trendline="ols",
    labels={
        "social_index": "Social Vulnerability Index",
        "working_age_health_index": "Working-Age Disability Index"
    },
    title="The Vicious Circle: Poverty → Disability (r=0.449)"
)

fig_main.update_layout(width=1000, height=700)
fig_main.show()

In [71]:
# --- 2.3 Surprising Finding: Social vs Child Health ---

fig_child = px.scatter(
    df_work,
    x="social_index",
    y="child_health_index",
    color=df_work["cluster"].astype(str),
    size="total_population",
    hover_name="settlement_name",
    trendline="ols",
    labels={
        "social_index": "Social Vulnerability Index",
        "child_health_index": "Child Disability Index"
    },
    title="Paradox: Poor Children Appear Healthier (r=-0.196) — Underdiagnosis?"
)

fig_child.update_layout(width=1000, height=700)
fig_child.show()


In [72]:
# --- 2.5 Future Projection: Working-Age vs Child Health ---

fig_future = px.scatter(
    df_work,
    x="working_age_health_index",
    y="child_health_index",
    color=df_work["cluster"].astype(str),
    size="total_population",
    hover_name="settlement_name",
    labels={
        "working_age_health_index": "Current: Working-Age Disability",
        "child_health_index": "Future: Child Disability"
    },
    title="Future Projection: Will It Get Better or Worse?"
)

# Add diagonal line (no change)
fig_future.add_shape(
    type="line",
    x0=-2, y0=-2, x1=2.5, y1=2.5,
    line=dict(color="gray", dash="dash")
)

fig_future.add_annotation(
    x=1.5, y=1.8,
    text="Above line = WORSENING",
    showarrow=False,
    font=dict(color="red")
)

fig_future.add_annotation(
    x=1.5, y=1.2,
    text="Below line = IMPROVING",
    showarrow=False,
    font=dict(color="green")
)

fig_future.update_layout(width=1000, height=700)
fig_future.show()

In [73]:
# --- 2.6 Cluster Comparison Bar Chart ---

cluster_means = df_work.groupby("cluster")[
    ["social_index", "working_age_health_index", "child_health_index", "elderly_health_index"]
].mean().round(3)

fig_bars = go.Figure()

colors = {"social_index": "blue", "working_age_health_index": "red",
          "child_health_index": "orange", "elderly_health_index": "purple"}

for col in cluster_means.columns:
    fig_bars.add_trace(go.Bar(
        name=col.replace("_", " ").title(),
        x=[f"Cluster {i}" for i in cluster_means.index],
        y=cluster_means[col]
    ))

fig_bars.update_layout(
    barmode="group",
    title="Cluster Profiles: All Indices Compared",
    yaxis_title="Index Value (standardized)",
    width=900,
    height=600
)

fig_bars.show()

In [74]:
# === CORRELATION BY CLUSTER ===

print("=" * 60)
print("CORRELATION: Social → Working-Age Health BY CLUSTER")
print("=" * 60)

for cluster in sorted(df_work["cluster"].unique()):
    subset = df_work[df_work["cluster"] == cluster]
    corr = subset["social_index"].corr(subset["working_age_health_index"])
    n = len(subset)
    print(f"Cluster {cluster}: r = {corr:.3f} (n={n})")

print("\n")

# === VISUALIZATION: Separate Trendlines by Cluster ===

fig_by_cluster = px.scatter(
    df_work,
    x="social_index",
    y="working_age_health_index",
    color=df_work["cluster"].astype(str),
    size="total_population",
    hover_name="settlement_name",
    trendline="ols",
    facet_col="cluster",
    labels={
        "social_index": "Social Vulnerability",
        "working_age_health_index": "Working-Age Disability"
    },
    title="Vicious Circle by Cluster: Different Patterns!"
)

fig_by_cluster.update_layout(width=1200, height=500)
fig_by_cluster.show()

CORRELATION: Social → Working-Age Health BY CLUSTER
Cluster 0: r = -0.165 (n=61)
Cluster 1: r = -0.013 (n=108)
Cluster 2: r = 0.438 (n=63)


